In [1]:
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np
from pathlib import Path
import math

def calculate_grid_size(n):
    """Calculate the optimal grid size for n images using predefined layouts."""
    # 预定义常见数量的最优布局，可以根据实际情况手动调整
    layouts = {
        # 1: (1, 1),
        # 2: (1, 2),
        # 3: (2, 2),
        # 4: (2, 2),
        4: (1, 4),
        # 5: (2, 3),
        5: (1, 5),
        # 6: (2, 3),
        # 6: (1, 6),
        # 7: (2, 4),
        # 7: (1, 7),
        # 8: (2, 4),
        # 9: (3, 3),
        # 10: (3, 4),
        # 11: (3, 4),
        # 12: (3, 4),
        # 可以继续添加更多...
        10: (4, 3),
        16: (6, 3),
    }
    
    # 如果数量在预定义布局中，直接返回
    if n in layouts:
        layout = layouts[n]
        print(f"Calculated grid size for {n} images: {layout}")
        return layout
    
    # 对于未预定义的数量，使用一个简单的计算方法
    rows = int(math.sqrt(n))
    cols = math.ceil(n / rows)
    layout = (rows, cols)
    # print(f"Calculated grid size for {n} images: {layout}")
    return layout

def load_image(path):
    """Load and convert image to numpy array."""
    img = Image.open(path)
    img = img.convert('RGB')
    return np.array(img)

def plot_comparison(folders_to_plot, image_name, compact_mode=False, figsize=(20, 12), dpi=300):
    """Plot comparison of same image from different folders."""
    n = len(folders_to_plot)
    rows, cols = calculate_grid_size(n)
    
    # 首先读取第一张图片来获取尺寸比例
    first_img = load_image(os.path.join(next(iter(folders_to_plot.values())), image_name))
    aspect_ratio = first_img.shape[1] / first_img.shape[0]  # 宽/高
    
    if compact_mode:
        # 根据图像比例调整figsize
        if figsize[0]/figsize[1] > (cols*aspect_ratio)/(rows):
            # 以高度为基准
            new_height = figsize[1]
            new_width = new_height * (cols*aspect_ratio)/rows
        else:
            # 以宽度为基准
            new_width = figsize[0]
            new_height = new_width * rows/(cols*aspect_ratio)
        
        fig = plt.figure(figsize=(new_width, new_height), frameon=False, dpi=dpi)
        gs = gridspec.GridSpec(rows, cols)
        gs.update(wspace=0, hspace=0, left=0, right=1, bottom=0, top=1)
    else:
        # 根据实际图像尺寸计算合适的子图大小
        img_height, img_width = first_img.shape[:2]
        
        # 计算合适的显示尺寸（英寸）
        # 假设理想的图像显示高度为4-6英寸
        target_height_inch = 5.0
        
        # 根据图像实际尺寸计算宽度
        img_width_inch = target_height_inch * aspect_ratio
        
        # 添加一些边距
        margin_factor = 1.1
        subplot_width = img_width_inch * margin_factor
        subplot_height = target_height_inch * margin_factor
        
        # 计算总的figure尺寸
        fig_width = subplot_width * cols
        fig_height = subplot_height * rows
        
        # 限制最大尺寸，避免过大的图像
        max_width = 30
        max_height = 20
        if fig_width > max_width:
            scale_factor = max_width / fig_width
            fig_width = max_width
            fig_height = fig_height * scale_factor
        if fig_height > max_height:
            scale_factor = max_height / fig_height
            fig_height = max_height
            fig_width = fig_width * scale_factor
            
        figsize = (fig_width, fig_height)
        fig = plt.figure(figsize=figsize, facecolor='white', dpi=dpi)
        gs = gridspec.GridSpec(rows, cols)
        gs.update(wspace=0.05, hspace=0.08)  # wspace控制水平间距，hspace控制垂直间距
    
    for idx, (folder_name, folder_path) in enumerate(folders_to_plot.items()):
        if idx < n:
            ax = plt.subplot(gs[idx])
            if folder_name == 'groundtruth' or folder_name == 'gt':
                image_name = image_name.replace('LR', 'HR')
                image_name = image_name.replace('_gauss_subsample', '')
            img_path = os.path.join(folder_path, image_name)
            img = load_image(img_path)
            
            # 使用 'equal' 而不是 'auto' 来保持原始比例
            ax.imshow(img, aspect='equal')
            ax.axis('off')
            
            if not compact_mode:
                ax.set_title(folder_name, pad=5)
            
            if compact_mode:
                ax.set_position([
                    ax.get_position().x0,
                    ax.get_position().y0,
                    ax.get_position().width,
                    ax.get_position().height
                ])
    
    if not compact_mode:
        plt.tight_layout()
    
    return fig

def plot_all_comparisons(folders_to_plot, save_dir=None, compact_mode=False):
    """Plot comparisons for all images."""
    original_folder = folders_to_plot['input']
    image_files = [f for f in os.listdir(original_folder) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    total_images = len(image_files)
    for idx, img_name in enumerate(image_files, 1):
        print(f"Processing image {idx}/{total_images}: {img_name}")
        fig = plot_comparison(folders_to_plot, img_name, compact_mode=compact_mode)
        
        if save_dir:
            mode_suffix = '_compact' if compact_mode else '_annotated'
            save_path = os.path.join(save_dir, f'comparison{mode_suffix}_{img_name}') # .replace("xijing_", "")
            Path(save_dir).mkdir(parents=True, exist_ok=True)
            
            if compact_mode:
                # 紧凑模式：完全无边距保存
                fig.savefig(save_path, 
                          bbox_inches='tight',
                          pad_inches=0,
                          facecolor='none',
                          transparent=True)
            else:
                fig.savefig(save_path, 
                          bbox_inches='tight',
                          facecolor='white')
            plt.close(fig)
        else:
            plt.show()

In [3]:
base_path = "/root/exp/us-hand-to-large"

# folders_to_plot = {
#     'original': os.path.join(base_path, 'datasets/all/test'),
#     'resnet_9block': os.path.join(base_path, 'results/test_1.resnet_9block_a401_x4'),
#     # # 'unet256_resize': os.path.join(base_path, 'results/test_2.raw_unet256_resize_256_3090'),
#     'unet_256': os.path.join(base_path, 'results/test_2.unet_256_3090_x256'),
#     # # 'hybrid_restormer': os.path.join(base_path, 'results/test_xxx.hybrid_restormer_1'),
#     # 'vq_resnet_3090_max256': os.path.join(base_path, 'results/test_3.vq_resnet_3090_x4_max256'),
#     # 'vq_resnet_3090_max512': os.path.join(base_path, 'results/test_3.vq_resnet_3090_x4_max512'),
#     # 'vq_resnet_3090_max1024': os.path.join(base_path, 'results/test_3.vq_resnet_3090_x4_max1024'),
#     # 'vq_resnet_patch256_overlap32': os.path.join(base_path, 'results/test_patch256_overlap32'),
#     # 'vq_resnet_patch256_overlap64': os.path.join(base_path, 'results/test_patch256_overlap64'),
#     # 'vq_resnet_patch256_overlap128': os.path.join(base_path, 'results/test_patch256_overlap128'),
#     # 'vq_resnet_patch512_overlap64': os.path.join(base_path, 'results/test_patch512_overlap64'),
#     # 'vq_resnet_patch512_overlap128': os.path.join(base_path, 'results/test_patch512_overlap128'),
#     # 'vq_resnet_patch512_overlap256': os.path.join(base_path, 'results/test_patch512_overlap256'),
#     'vq_resnet_a6000_epoch185_max256': os.path.join(base_path, 'results/test_3.vq_resnet_a6000_epoch185_x4_max256'),
#     'vq_resnet_a6000_epoch185_max512': os.path.join(base_path, 'results/test_3.vq_resnet_a6000_epoch185_x4_max512'),
#     'vq_resnet_a6000_epoch185_max1024': os.path.join(base_path, 'results/test_3.vq_resnet_a6000_epoch185_x4_max1024'),
#     'vq_resnet_concat_paired_max256': os.path.join(base_path, 'results/test_grayscale_x4_max256'),
#     'vq_resnet_concat_paired_max512': os.path.join(base_path, 'results/test_grayscale_x4_max512'),
#     'vq_resnet_concat_paired_max1024': os.path.join(base_path, 'results/test_grayscale_x4_max1024'),
#     'vq_hybrid_restormer': os.path.join(base_path, 'results/test_xxx.hybrid_restormer_1'),
# }

# folders_to_plot = {  # gt
#     'input': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_LR'),
#     "new infer fix256back": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_dual_AtoB_fix256back",
#     "new infer resize256": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_dual_AtoB_resize256",
#     "new infer noresize": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_dual_AtoB_noresize",
#     # 'AtoA': "/root/exp/us-hand-to-large/results/xijing_test_dual_AtoA_LR_SR_x4_max256",
#     'old infer ': "/root/exp/us-hand-to-large/results/xijing_test_dual_AtoB_LR_SR_x4_max256",
#     'old model': "/root/exp/us-hand-to-large/results/xijing_test_LR_SR_x4_max256",
#     'groundtruth': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_HR'),
# }
# folders_to_plot = {  # gt
#     'input': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_LR'),
#     "fix Resize256 Infer Restore": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_vqdualv1_AtoB_fulldata_fixResize256Restore",
#     "fix Resize512 Infer Restore": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_vqdualv1_AtoB_fulldata_fixResize512Restore",
#     "std Load286 Resize256 Infer NoRestore": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_vqdualv1_AtoB_fulldata_stdLoad286Crop256",
#     "flex NoResize": "/root/exp/us-hand-to-large/results/infer_new/xijing_test_vqdualv1_AtoB_fulldata_flexNoResize",
#     # 'AtoA': "/root/exp/us-hand-to-large/results/xijing_test_dual_AtoA_LR_SR_x4_max256",
#     # 'old infer ': "/root/exp/us-hand-to-large/results/xijing_test_dual_AtoB_LR_SR_x4_max256",
#     # 'old model': "/root/exp/us-hand-to-large/results/xijing_test_LR_SR_x4_max256",
#     'groundtruth': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_HR'),
# }

# folders_to_plot = {  # 配对，real-vanilla-vqresnet-vqdualv0-vqdualv1-vqdualv1
#     'input': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_LR_crop_gray'),
#     'real-esrgan': '/root/exp/us-hand-to-large/results/comparative-exp/real_esrgan/test_cropdata_train0_SR',
#     'vanilla_cyclegan': '/root/exp/us-hand-to-large/results/comparative-exp/vanilla_cyclegan/xijing_test_vanilla_cyclegan_cropdata_flexNoResize',
#     'unsb': '/root/exp/us-hand-to-large/results/comparative-exp/unsb/unsb_semi_paired_test_gray/fake_1',
#     'vqresnet': os.path.join(base_path, 'results/infer_new/xijing_test_vqresnet_AtoB_cropdata_flexNoResize'),
#     'vqdualv0': os.path.join(base_path, 'results/infer_new/xijing_test_vqdualv0_AtoB_cropdata_flexNoResize'),
#     'vqdualv1': os.path.join(base_path, 'results/infer_new/xijing_test_vqdualv1_AtoB_cropdata_flexNoResize'),
#     'vqdualv2': os.path.join(base_path, 'results/infer_new/xijing_test_vqdualv2Paired10_AtoB_cropdata_flexNoResize'),
#     'vqdualv3': os.path.join(base_path, 'results/infer_new/xijing_test_vqdualv3Paired10Semipaired10_AtoB_cropdata_flexNoResize'),
#     'vqdualv3_dim64': os.path.join(base_path, 'results/infer_new/xijing_test_vqdualv3_dim64_Paired10Semipaired10_AtoB_cropdata_flexNoResize'),
#     'contmixv0': os.path.join(base_path, 'results/infer_new/xijing_test_contmixv0_k13s5_k7s5'),
#     'contmixv1': os.path.join(base_path, 'results/infer_new/xijing_test_contmixv1_k7s3_k13s5'),
#     'gt': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_HR_crop_gray'),
# }

folders_to_plot = {  # 展示
    'input': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_LR_crop_gray'),
    'sr': os.path.join(base_path, 'results/infer_new/xijing_test_contmixv0_k13s5_k7s5'),
    'gt': os.path.join(base_path, 'datasets/xijing_split/test/test_semi_paired_HR_crop_gray'),
}


# folders_to_plot = {  # 非配对
#     'input': os.path.join(base_path, 'datasets/xijing_split/trainA_crop_gray'),
#     'real-esrgan': '/root/exp/us-hand-to-large/results/comparative-exp/real_esrgan/xijing_trainACropGray_RealESRGAN',
#     'vanilla_cyclegan': '/root/exp/us-hand-to-large/results/comparative-exp/vanilla_cyclegan/xijing_trainACropGray_vanillaCyclegan_flexNoResize',
#     'unsb': '/root/exp/us-hand-to-large/results/comparative-exp/unsb/unsb_unpaired_test_gray/fake_1',
#     'vqresnet': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqresnet_flexNoResize'),
#     'vqdualv0': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqdualv0_AtoB_flexNoResize'),
#     'vqdualv1': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqdualv1_AtoB_flexNoResize'),
#     'vqdualv2': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqdualv2Paired10_AtoB_flexNoResize'),
#     'vqdualv3': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqdualv3Paired10Semipaired10_AtoB_flexNoResize'),
#     'vqdualv3_dim64': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_vqdualv3_dim64_Paired10Semipaired10_AtoB_flexNoResize'),
#     'contmixv0': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_contmixv0_k13s5_k7s5'),
#     'contmixv1': os.path.join(base_path, 'results/infer_new/xijing_trainACropGray_contmixv1_k7s3_k13s5'),
# }

# folders_to_plot = {
#     'input': os.path.join(base_path, 'datasets/split/test/test_only_LR'),
#     # 'resnet_9block': os.path.join(base_path, 'results/test_1.resnet_9block_a401_x4'),
#     # 'unet_256': os.path.join(base_path, 'results/test_2.unet_256_3090_x256'),
#     'output': "/root/Lecter/cyclegan-exp/us-hand-to-large/results/results_semi_paired/test_6.vq_resnet_v1_semi_un_paired_onlyLR_x4max256",
# }

# folders_to_plot = {  # 张博
#     'input': "/root/exp/us-hand-to-large/datasets/zhang/origin/t1090000101al-dir_shrink6",
#     'output': "/root/exp/us-hand-to-large/results/zhang/t1090000101al-dir_shrink6_sr",
#     'groundtruth': "/root/exp/us-hand-to-large/datasets/zhang/origin/t1090000101al-dir",
# }

# 使用示例
# 定义保存目录（可选）
# 绘制所有比较图
# save_dir = os.path.join(base_path, 'results/comparisons/xijing_trainACropGray_12method/label')
# save_dir = os.path.join(base_path, 'results/comparisons/xijing_13method_cropdata/label')
save_dir = os.path.join(base_path, 'results/comparisons/contmixv0/label')
plot_all_comparisons(folders_to_plot, save_dir, compact_mode=False)

# # 如果只想查看单张图片的比较
# image_files = [f for f in os.listdir(folders_to_plot['input']) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
# image_name = image_files[1]  # 读取列表中的第一张图片
# plot_comparison(folders_to_plot, image_name, compact_mode=False, dpi=600)
# plt.show()

Processing image 1/53: xijing_LR_314.png


/tmp/ipykernel_99739/2928494460.py:138: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Processing image 2/53: xijing_LR_335.png
Processing image 3/53: xijing_LR_71.png
Processing image 4/53: xijing_LR_498.png
Processing image 5/53: xijing_LR_15.png
Processing image 6/53: xijing_LR_83.png
Processing image 7/53: xijing_LR_235.png
Processing image 8/53: xijing_LR_379.png
Processing image 9/53: xijing_LR_38.png
Processing image 10/53: xijing_LR_164.png
Processing image 11/53: xijing_LR_324.png
Processing image 12/53: xijing_LR_129.png
Processing image 13/53: xijing_LR_117.png
Processing image 14/53: xijing_LR_11.png
Processing image 15/53: xijing_LR_7.png
Processing image 16/53: xijing_LR_349.png
Processing image 17/53: xijing_LR_74.png
Processing image 18/53: xijing_LR_469.png
Processing image 19/53: xijing_LR_451.png
Processing image 20/53: xijing_LR_506.png
Processing image 21/53: xijing_LR_118.png
Processing image 22/53: xijing_LR_428.png
Processing image 23/53: xijing_LR_120.png
Processing image 24/53: xijing_LR_310.png
Processing image 25/53: xijing_LR_290.png
Processi